In [ ]:
# Step 1: 安装依赖（如果需要）
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "mcp", "openai", "python-dotenv", "-q"])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径（动态识别）
import sys
from pathlib import Path

cwd = Path.cwd()

# 如果在 notebooks 目录，项目根目录是上一级
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    # 向上查找 pyproject.toml
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

# MCP (Model Context Protocol) 基础教程

本教程帮助你理解 MCP 的核心概念和使用方式。

## 学习目标

1. 理解 MCP 是什么，解决什么问题
2. 掌握 MCP 三大原语：Tool、Resource、Prompt
3. 学会使用 MCP 客户端调用工具
4. 理解 MCP 与 LLM 的结合方式

## 1. 什么是 MCP？

MCP (Model Context Protocol) 是 Anthropic 推出的**开放协议**，用于连接 AI 助手与外部系统。

### 解决的问题

```
传统方式：
每个 LLM 应用都要为每个工具写专门的集成代码

LLM App A ──→ Tool 1 (专用接口)
         ├──→ Tool 2 (专用接口)
         └──→ Tool 3 (专用接口)

MCP 方式：
统一的协议，一次集成，到处使用

LLM App A ─┐
LLM App B ─┼─→ MCP 协议 ─→ Tool Server 1
LLM App C ─┘              └→ Tool Server 2
```

### 核心优势

1. **标准化**: 统一的工具接口规范
2. **可复用**: 工具服务器可被多个应用使用
3. **可扩展**: 方便添加新的工具和能力

## 2. MCP 三大原语

MCP 定义了三种核心原语（Primitive）：

| 原语 | 用途 | 示例 |
|------|------|------|
| **Tool** | 可调用的函数 | 搜索、计算、查询数据库 |
| **Resource** | 可读取的数据 | 文件内容、数据库记录 |
| **Prompt** | 预定义的提示模板 | 代码审查模板、翻译模板 |

In [ ]:
# 导入 MCP 相关类型
from rogue_rabbit.contracts import (
    MCPTool,
    MCPToolInputSchema,
    MCPToolResult,
    MCPResource,
    MCPPrompt,
    MCPServerConfig,
    MCPTransportType,
    MockMCPClient,
)

print("导入成功！")

## 3. Tool（工具）- MCP 的核心

Tool 是 MCP 最常用的原语，允许 LLM 调用外部函数。

In [ ]:
# 定义一个简单的工具
weather_tool = MCPTool(
    name="get_weather",
    description="获取指定城市的当前天气信息",
    input_schema=MCPToolInputSchema(
        properties={
            "city": {
                "type": "string",
                "description": "城市名称，如 '北京'、'上海'"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "温度单位"
            }
        },
        required=["city"]
    )
)

print(f"工具名称: {weather_tool.name}")
print(f"工具描述: {weather_tool.description}")
print(f"必需参数: {weather_tool.input_schema.required}")

### JSON Schema 详解

工具的 `input_schema` 使用 JSON Schema 描述参数：

```json
{
  "type": "object",
  "properties": {
    "参数名": {
      "type": "string/number/boolean/...",
      "description": "参数说明"
    }
  },
  "required": ["必需参数列表"]
}
```

LLM 会根据这个 schema 生成正确的参数格式。

## 4. 使用 Mock 客户端学习

在没有真实 MCP 服务器的情况下，我们可以使用 MockMCPClient 来学习调用流程。

In [ ]:
# 创建多个工具
tools = [
    MCPTool(
        name="calculator",
        description="执行数学计算，支持加减乘除",
        input_schema=MCPToolInputSchema(
            properties={
                "expression": {
                    "type": "string",
                    "description": "数学表达式，如 '2+3*4'"
                }
            },
            required=["expression"]
        )
    ),
    MCPTool(
        name="search",
        description="在知识库中搜索信息",
        input_schema=MCPToolInputSchema(
            properties={
                "query": {
                    "type": "string",
                    "description": "搜索关键词"
                }
            },
            required=["query"]
        )
    ),
]

# 预设工具返回结果
tool_results = {
    "calculator": MCPToolResult(content="14"),
    "search": MCPToolResult(content="Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。"),
}

print(f"定义了 {len(tools)} 个工具")

In [ ]:
import asyncio

# 创建 Mock 客户端
client = MockMCPClient(tools=tools, tool_results=tool_results)

# 使用 async with 管理连接
async def demo_list_tools():
    async with client:
        # 列出所有可用工具
        available_tools = await client.list_tools()
        print("可用的工具：")
        for tool in available_tools:
            print(f"  - {tool.name}: {tool.description}")

await demo_list_tools()

In [ ]:
# 调用工具
async def demo_call_tool():
    async with client:
        # 调用计算器
        result = await client.call_tool("calculator", {"expression": "2+3*4"})
        print(f"计算器结果: {result.text}")
        
        # 调用搜索
        result = await client.call_tool("search", {"query": "Python"})
        print(f"搜索结果: {result.text}")

await demo_call_tool()

## 5. Resource（资源）

Resource 用于暴露可读取的数据，如文件、数据库记录等。

In [ ]:
# 定义资源
resources = [
    MCPResource(
        uri="file:///config.json",
        name="配置文件",
        description="应用程序配置",
        mime_type="application/json"
    ),
    MCPResource(
        uri="database:///users",
        name="用户表",
        description="用户信息数据库表"
    ),
]

for res in resources:
    print(f"资源: {res.name}")
    print(f"  URI: {res.uri}")
    print(f"  描述: {res.description}")
    print()

## 6. MCP 服务器配置

MCP 支持两种传输方式：

In [ ]:
# STDIO 配置 - 本地进程通信
stdio_config = MCPServerConfig(
    name="local-tools",
    transport=MCPTransportType.STDIO,
    command="python",
    args=["-m", "my_mcp_server"],
    env={"DEBUG": "1"}  # 可选的环境变量
)

print("STDIO 配置:")
print(f"  命令: {stdio_config.command} {' '.join(stdio_config.args)}")

# HTTP 配置 - 远程服务通信
http_config = MCPServerConfig(
    name="remote-tools",
    transport=MCPTransportType.STREAMABLE_HTTP,
    url="http://localhost:8000/mcp"
)

print("\nHTTP 配置:")
print(f"  URL: {http_config.url}")

### 传输方式对比

| 方式 | 适用场景 | 优点 | 缺点 |
|------|----------|------|------|
| **STDIO** | 本地工具、开发调试 | 简单、安全 | 只能本地 |
| **HTTP** | 远程服务、生产环境 | 可分布式 | 需要网络配置 |

## 7. LLM + MCP = Agent

将 LLM 和 MCP 工具结合，就构成了 Agent 的核心能力。

In [ ]:
# Agent 工作流程图
print("""
Agent 工作流程:
================

用户问题
    │
    ▼
┌─────────────┐
│    LLM      │ ◄─── 思考：需要什么工具？
└─────────────┘
    │
    ▼ 决定调用工具
┌─────────────┐
│ MCP Client  │ ◄─── 执行工具调用
└─────────────┘
    │
    ▼ 获取结果
┌─────────────┐
│    LLM      │ ◄─── 根据结果继续思考
└─────────────┘
    │
    ▼
最终答案

这就是 ReAct (Reasoning + Acting) 模式！
""")

In [ ]:
# 简单的 Agent 示例
from rogue_rabbit.contracts import Message, Role, MockLLMClient

# 模拟 LLM 的决策过程
def simulate_agent_thinking(question: str, tools: list[MCPTool]) -> str:
    """
    模拟 Agent 的思考过程
    真正的 Agent 会调用 LLM 来做这个决策
    """
    print(f"\n问题: {question}")
    print(f"\n可用工具: {[t.name for t in tools]}")
    
    # 模拟 LLM 决策
    if "计算" in question or "*" in question:
        print("\n[LLM 思考] 这是一个数学问题，需要使用计算器")
        return "calculator"
    elif "搜索" in question or "什么是" in question:
        print("\n[LLM 思考] 需要查找信息，使用搜索工具")
        return "search"
    else:
        print("\n[LLM 思考] 不需要工具，直接回答")
        return None

# 演示
simulate_agent_thinking("计算 123 * 456", tools)
simulate_agent_thinking("什么是 Python？", tools)

## 8. 实际应用示例

让我们看一个完整的 Agent 执行流程。

In [ ]:
async def simple_agent_demo():
    """简单的 Agent 演示"""
    
    # 准备工具
    tools = [
        MCPTool(
            name="get_time",
            description="获取当前时间",
            input_schema=MCPToolInputSchema(
                properties={},
                required=[]
            )
        ),
        MCPTool(
            name="calculate",
            description="计算数学表达式",
            input_schema=MCPToolInputSchema(
                properties={"expr": {"type": "string"}},
                required=["expr"]
            )
        ),
    ]
    
    tool_results = {
        "get_time": MCPToolResult(content="2024-03-25 15:30:00"),
        "calculate": MCPToolResult(content="42"),
    }
    
    client = MockMCPClient(tools=tools, tool_results=tool_results)
    
    # Agent 执行
    question = "现在几点了？另外帮我算一下 6*7"
    print(f"用户: {question}\n")
    
    async with client:
        # 第一步：获取时间
        print("[Agent] 需要调用 get_time 工具")
        time_result = await client.call_tool("get_time", {})
        print(f"[工具结果] 当前时间: {time_result.text}\n")
        
        # 第二步：计算
        print("[Agent] 需要调用 calculate 工具")
        calc_result = await client.call_tool("calculate", {"expr": "6*7"})
        print(f"[工具结果] 计算结果: {calc_result.text}\n")
        
        # 最终回答
        print("[Agent 最终回答]")
        print(f"现在是 {time_result.text}，6×7={calc_result.text}")

await simple_agent_demo()

## 总结

### 核心概念

1. **MCP**: 标准化的工具协议，连接 LLM 与外部系统
2. **Tool**: 可调用的函数，LLM 决定何时调用
3. **Resource**: 可读取的数据源
4. **Prompt**: 预定义的提示模板

### 关键理解

- MCP 让工具集成变得标准化
- Agent = LLM + Tools（MCP 提供工具接口）
- ReAct 模式：思考 → 行动 → 观察 → 循环

### 下一步

- 运行 `experiments/04_mcp_basic.py` 和 `05_mcp_with_llm.py`
- 尝试连接真实的 MCP 服务器
- 阅读源码 `contracts/mcp.py` 和 `adapters/mcp_client.py`